In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [10]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions
)


In [9]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about running **Ollama locally**.\n\nThe closest related note says you can run the **course locally** instead of Codespaces if you’re comfortable setting up Python, `uv`, Jupyter, Docker, and any other needed tools. If you do that, document your setup and keep it reproducible.\n\nIf you meant a specific setup for Ollama, I’d need the relevant FAQ entry.'

In [10]:
resultados_certo = assistant.search("How do I run Ollama locally?")
resultados_errado = assistant.search("How do I run Olama locally?")

print("Resultados com 'Ollama' (correto):")
print(len(resultados_certo))
print(resultados_certo)

print("\nResultados com 'Olama' (erro de digitação):")
print(len(resultados_errado))
print(resultados_errado)

Resultados com 'Ollama' (correto):
5
[{'id': '1d0b969028', 'course': 'llm-zoomcamp', 'section': 'Module 1: RAG', 'question': 'Ollama: How to install Ollama?', 'answer': 'First, install Ollama by visiting [https://ollama.com/download](https://ollama.com/download) and choosing your operating system:\n\n- **macOS**: Download the `.pkg` and install it.\n- **Windows**: Download the `.msi` and install it.\n- **Linux**: Run the following command in the terminal:\n\n  ```bash\n  curl -fsSL https://ollama.com/install.sh | sh\n  ```\n\nOnce installed, open a terminal and type:\n\n```bash\nollama run llama3\n```\n\nThis command will:\n\n- Download the LLaMA 3 model (~4GB).\n- Start the model locally.\n- Open a chat-like interface where you can type questions.\n\nTo test the Ollama local server, run the following command:\n\n```bash\ncurl http://localhost:11434\n```\n\nYou should receive a response similar to:\n\n```json\n{"models": [...]}  \n```\n\nThen, install the Python client with:\n\n```bash

In [4]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—if the course is still open, you can usually join it.\n\nTo confirm, I’d need a bit more information:\n- the course name\n- where it’s hosted\n- whether you mean enrolling, transferring in, or attending as a guest\n\nIf you want, I can help you figure out:\n1. whether enrollment is still open,\n2. how to register, and\n3. what to do if the course has already started.'

In [6]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [8]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course"}', call_id='call_p4UEueiFW3DX4HBShtxT5RQL', name='search', type='function_call', id='fc_07788289f90251bb006a389b112958819d9bc6a19157eaef35', namespace=None, status='completed')]

In [14]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)
print(result_json)

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "bd31146b0e",
    "course": "llm-zoomcamp",
    "section": "General Course

In [15]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [17]:
print(messages)

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'}, ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I join course"}', call_id='call_p4UEueiFW3DX4HBShtxT5RQL', name='search', type='function_call', id='fc_07788289f90251bb006a389b112958819d9bc6a19157eaef35', namespace=None, status='completed'), {'type': 'function_call_output', 'call_id': 'call_p4UEueiFW3DX4HBShtxT5RQL', 'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "69d122f12e",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",\n    "

In [18]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can still join the course.\n\nIf you want a certificate, you’ll need to submit your project while submissions are still open.'